In [50]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Explicit path: the kernel's working directory may not be this folder.
# override=True replaces any empty GROQ_API_KEY already set in this kernel.
load_dotenv(Path("/Users/jobayerahmed/Documents/ML/langchain/ollama/.env"), override=True)
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found - check .env"

model = ChatGroq(model_name="openai/gpt-oss-120b")

In [51]:
from langchain_core.messages import HumanMessage

In [52]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are helpful assistant. Answer all the question to the best of your ability'),
     MessagesPlaceholder(variable_name="message")
])
chain = prompt|model

In [53]:
chain.invoke({"message": [HumanMessage(content="My name is jobayer ahmed")]})

AIMessage(content='Nice to meet you, Jobayer Ahmed! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "My name is jobayer ahmed". Likely they are just stating name, maybe expecting a response. Should respond politely, maybe ask how can I help. Use appropriate tone.'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 96, 'total_tokens': 162, 'completion_time': 0.139242293, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.004481748, 'prompt_tokens_details': None, 'queue_time': 0.212092249, 'total_time': 0.143724041}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498a', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16961-6fcc-7f10-a6f2-9aba47ba2ccc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 66, 'total_tokens': 162, 'output_token_details': {'reasoning': 41}})

## Message history

In [54]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [55]:
# One chat history per session_id (separate name so `chain` isn't overwritten)
store = {}

def get_message_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [56]:
# Wrap the chain: past messages of the session get added in front of "message"
with_message_history = RunnableWithMessageHistory(
    chain,
    get_message_history,
    input_messages_key="message",
)
config = {"configurable": {"session_id": "chat1"}}

### With history

In [57]:
response = with_message_history.invoke(
    {"message": [HumanMessage(content="Hi, my name is Jobayer Ahmed")]},
    config=config,
)
response.content

'Hello Jobayer Ahmed! Nice to meet you. How can I assist you today?'

In [58]:
# Same session -> the model remembers the name
response = with_message_history.invoke(
    {"message": [HumanMessage(content="What is my name?")]},
    config=config,
)
response.content

'Your name is Jobayer\u202fAhmed.'

### Without history

In [59]:
# Plain chain has no memory -> the model doesn't know the name
response = chain.invoke({"message": [HumanMessage(content="What is my name?")]})
response.content

'I’m sorry, but I don’t have any information about your name. If you’d like to share it, feel free to let me know!'

In [60]:
# New session_id starts with an empty history -> doesn't know the name either
response = with_message_history.invoke(
    {"message": [HumanMessage(content="What is my name?")]},
    config={"configurable": {"session_id": "chat2"}},
)
response.content

'I don’t actually know your name—unless you’ve shared it with me earlier in our conversation, I don’t have that information. If you’d like me to address you by name, just let me know what it is!'

In [61]:
# Messages stored for chat1
store["chat1"].messages

[HumanMessage(content='Hi, my name is Jobayer Ahmed', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello Jobayer Ahmed! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hi, my name is Jobayer Ahmed". Likely they are greeting and introducing themselves. The assistant should respond politely, perhaps ask how can help. There\'s no policy violation. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 97, 'total_tokens': 167, 'completion_time': 0.15034946, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.010541476, 'prompt_tokens_details': None, 'queue_time': 0.400072979, 'total_time': 0.160890936}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16961-71bc-7a20-abea-4afef5f2ffd8-0', tool_calls=[], invalid_

# Managing chat history

In [66]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter="approximate",
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [67]:
from langchain_core.messages import AIMessage

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage (content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage (content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage (content= "thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content= "having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]